# Load Dataset

In [2]:
import numpy as np 
import pandas as pd

import seaborn as sns 
import matplotlib.pyplot as plt  

In [3]:
df = sns.load_dataset("titanic")
df.shape 

(891, 15)

In [4]:
df.columns 

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')

In [5]:
# 필요없는 컬럼 제거
df.drop(
    ['class', 'embark_town', 'alive'], axis=1, inplace=True
)

df.shape 

(891, 12)

## Target

In [6]:
# 각 class간의 비율 확인 
df['survived'].value_counts() / df.shape[0]

survived
0    0.616162
1    0.383838
Name: count, dtype: float64

## 평가 & 학습 데이터셋 

In [7]:
from common.utils import train_test_split_by_target

train, test = train_test_split_by_target(df)

train.shape, test.shape 

((668, 12), (223, 12))

# EDA

In [8]:
train.info()

<class 'pandas.DataFrame'>
Index: 668 entries, 486 to 821
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   survived    668 non-null    int64   
 1   pclass      668 non-null    int64   
 2   sex         668 non-null    str     
 3   age         537 non-null    float64 
 4   sibsp       668 non-null    int64   
 5   parch       668 non-null    int64   
 6   fare        668 non-null    float64 
 7   embarked    666 non-null    str     
 8   who         668 non-null    str     
 9   adult_male  668 non-null    bool    
 10  deck        145 non-null    category
 11  alone       668 non-null    bool    
dtypes: bool(2), category(1), float64(2), int64(4), str(3)
memory usage: 54.5 KB


# Data Preprocessing

## data cleaning

In [9]:
# row 중복 제거 
print(f"before: {train.shape}")

# 평가는 학습을 하는 데이터가 아니기 때문에 row 중복 제거 제외
train.drop_duplicates(inplace=True)
print(f"after: {train.shape}")

before: (668, 12)
after: (604, 12)


In [10]:
# 결측치 확인 및 제거 
null_tr = train.isnull().sum()

(null_tr[null_tr > 0] / train.shape[0]).sort_values(ascending=False)

deck        0.759934
age         0.142384
embarked    0.003311
dtype: float64

In [11]:
print(f"before: {train.isnull().sum().sum()} / {test.isnull().sum().sum()}")

for tmp in [train, test]:
    null_tmp = tmp.isnull().sum()
    null_cols = null_tmp[null_tmp > 0].index

    for col in null_cols:
        try:
            tmp[col] = tmp[col].fillna(train[col].mean())

        except:
            tmp[col] = tmp[col].fillna(train[col].mode().values[0])

print(f"after: {train.isnull().sum().sum()} / {test.isnull().sum().sum()}")

before: 547 / 211
after: 0 / 0


# Modeling

In [12]:
from common.modeling import Modeling

## Check Dataset before Modeling

In [13]:
modeling = Modeling(
    x_tr=train.drop(['survived'], axis=1),
    y_tr=train['survived']
)

In [14]:
modeling.fit_evaluation(
    x_te = test.drop(['survived'], axis=1),
    y_te = test['survived']
)

In [15]:
modeling.get_best_model()

{'model_name': 'CatBoostClassifier',
 'hpo': {'verbose': 0,
  'cat_features': ['pclass',
   'sex',
   'embarked',
   'who',
   'adult_male',
   'deck',
   'alone']},
 'train_score': 0.8386643711489671,
 'test_score': 0.787599728399253,
 'score_type': 'auc'}

# Evaluation

In [18]:
print(
    f'훈련용 평가지표: {modeling.get_best_model()['train_score']} / 테스트용 평가지표: {modeling.get_best_model()['test_score']}'
)

훈련용 평가지표: 0.8386643711489671 / 테스트용 평가지표: 0.787599728399253


In [19]:
from common.evaluation_plots import show_norm_conf_mx

pred = modeling.predict_by_best_model(test.drop(['survived'], axis=1))

show_norm_conf_mx(y=test['survived'], pred=pred)

ModuleNotFoundError: No module named 'common.evaluation_plots'

In [20]:
from common.modeling import BoostModelType

BoostModelType.show_plot_importance(modeling.get_best_model()['model'])

AttributeError: type object 'BoostModelType' has no attribute 'show_plot_importance'